# 04. Informational Offer Analysis

## Goal
Find out whether customers make purchases after receiving informational offers, and whether viewing the offer matters.

## Why this needs its own analysis
BOGO and discount offers have an `offer completed` event, so success is clear. Informational offers offer no reward and have **no completion event**, so the funnel cannot measure their success. Here, success is approximated by purchase activity inside the offer window.

## Inputs
- `offer_funnel.csv` (one row per offer received, with view times)
- `events_clean.csv` (for transactions)
- `offers_clean.csv` (for offer details and channels)

## Output
- `informational_offer_analysis.csv`

## Method note
Transactions in this dataset carry no `offer_id`, so any link between a transaction and an informational offer is inferred from customer and time window. The results show **association, not causation**.

## 1. Load data and keep informational offers only
The funnel file contains all offer types. We keep only informational receipts, which should be 13,300 rows.

In [ ]:
import pandas as pd
import numpy as np



In [ ]:
import pandas as pd
import ast

funnel = pd.read_csv('../data/cleaned/offer_funnel.csv')
events = pd.read_csv('../data/cleaned/events_clean.csv')
offers = pd.read_csv('../data/cleaned/offers_clean.csv')

info = funnel[funnel['offer_type'].eq('informational')].copy()
n_info = len(info)
print("Informational receipts:", n_info)      # expect 13,300

## 2. The two informational offers
There are only two informational offers. Before analysing anything, we look at how they differ (duration and channels), because these differences can affect how often customers view them.

In [ ]:
# channels was saved as text like "['email', 'mobile']"; literal_eval turns it back into a real list
offers['channels_list'] = offers['channels'].apply(ast.literal_eval)

info_offers = offers[offers['offer_type'].eq('informational')][['offer_id', 'duration', 'channels_list']]
info_offers

## 3. Link transactions to each offer receipt
For each received informational offer, we count the customer's transactions between the time they received it and the time it expired.

We calculate:
- `tx_in_window`: number of transactions inside the offer window
- `spend_in_window`: total spent inside the offer window
- `tx_after_view`: transactions at or after the customer's first view of the offer

**Example:** a customer receives an offer at hour 0 (3-day window, expires at hour 72), views it at hour 10, and buys at hours 30 and 90. `tx_in_window` = 1 (hour 30 only, since hour 90 is after expiry), and `tx_after_view` = 1.

**Assumption:** a transaction in the same hour as the first view counts as "after view" (`>=`), because the exact order inside an hour is unknown.

In [ ]:
tx = (events.loc[events['event'].eq('transaction'), ['customer_id', 'time', 'amount']]
      .rename(columns={'time': 'tx_time'}))

# pair every receipt with every transaction by the same customer, keep those inside the window
m = info[['received_event_id', 'customer_id', 'received_time', 'window_end', 'first_view_time']].merge(
    tx, on='customer_id')
m = m[(m['tx_time'] >= m['received_time']) & (m['tx_time'] <= m['window_end'])]

in_window = m.groupby('received_event_id').agg(
    tx_in_window=('amount', 'size'), spend_in_window=('amount', 'sum'))

after_view = (m[m['first_view_time'].notna() & (m['tx_time'] >= m['first_view_time'])]
              .groupby('received_event_id').size().rename('tx_after_view'))

info = (info.merge(in_window, on='received_event_id', how='left')
            .merge(after_view, on='received_event_id', how='left')
            .fillna({'tx_in_window': 0, 'spend_in_window': 0, 'tx_after_view': 0}))

info['any_tx_in_window'] = info['tx_in_window'] > 0
info['tx_after_view_flag'] = info['tx_after_view'] > 0

# merges must not add or remove receipts
assert len(info) == n_info

## 4. Viewed vs not viewed
Both groups received the same kind of offer and have the same purchase window. We compare purchase activity across the **full window** (receipt to expiry) for both, so the comparison is fair. Using the time after the view would give viewers a shorter window.

We also calculate the KPI "viewed offers followed by a transaction": of the viewed offers, how many were followed by a purchase.

In [ ]:
compare = info.groupby('viewed').agg(
    receipts=('viewed', 'size'),
    pct_with_transaction=('any_tx_in_window', 'mean'),
    avg_transactions=('tx_in_window', 'mean'),
    avg_spend=('spend_in_window', 'mean'),
    median_spend=('spend_in_window', 'median'))
display(compare.round(3))

viewed_n = info['viewed'].sum()
followed_n = info['tx_after_view_flag'].sum()
print(f"Viewed offers followed by a transaction: {followed_n} of {viewed_n} ({followed_n / viewed_n:.1%})")

## 5. Check for other offers running at the same time
A customer might buy because of a BOGO or discount offer that was active at the same time, not because of the informational one. We flag informational receipts where another offer's window overlapped, then repeat the viewed vs not viewed comparison for each group.

If the gap between viewers and non-viewers remains when no other offer was active, the result is less likely to be caused by the other offers.

In [ ]:
others = (funnel[funnel['offer_type'].ne('informational')]
          [['customer_id', 'received_time', 'window_end']]
          .rename(columns={'received_time': 'o_start', 'window_end': 'o_end'}))

j = info[['received_event_id', 'customer_id', 'received_time', 'window_end']].merge(others, on='customer_id')
j = j[(j['o_start'] <= j['window_end']) & (j['o_end'] >= j['received_time'])]   # windows overlap
info['other_offer_active'] = info['received_event_id'].isin(j['received_event_id'])

print(f"Share with another offer active: {info['other_offer_active'].mean():.1%}")
info.groupby(['other_offer_active', 'viewed']).agg(
    receipts=('viewed', 'size'),
    pct_with_transaction=('any_tx_in_window', 'mean'),
    avg_spend=('spend_in_window', 'mean')).round(3)

## 6. Breakdown by offer
The two informational offers differ in duration and channels. We compare view rate and purchase rate for each, to see whether the offer that gets viewed more also gets more purchases. If viewing drives purchasing, we would expect it to.

In [ ]:
by_offer = info.groupby(['offer_id', 'duration']).agg(
    receipts=('viewed', 'size'),
    view_rate=('viewed', 'mean'),
    pct_with_transaction=('any_tx_in_window', 'mean')).reset_index()
by_offer = by_offer.merge(info_offers[['offer_id', 'channels_list']], on='offer_id')
display(by_offer.round(3))

info.groupby(['offer_id', 'viewed']).agg(
    receipts=('viewed', 'size'),
    pct_with_transaction=('any_tx_in_window', 'mean'),
    avg_spend=('spend_in_window', 'mean')).round(3)

## 8. Save the results

In [ ]:
info.to_csv('../data/cleaned/informational_offer_analysis.csv', index=False)
print("Saved:", info.shape)

## 7. Findings, assumptions and limitations

### What the data shows
- Customers who viewed an informational offer had a transaction in the offer window more often than those who did not (68.7% vs 53.1%) and spent more on average ($17.92 vs $10.86).
- 61.2% of viewed offers (5,252 of 8,585) were followed by a transaction.
- The gap remains when no other offer was active (63.8% vs 41.0% with a transaction), so overlapping offers do not fully explain it.
- The two offers have very different view rates (47.7% vs 81.5%) but almost identical purchase rates (62.8% vs 63.6%). The higher-viewed offer is the one sent through the social channel.

### Interpretation (not proven by the data)
- Customers who open offers are probably more engaged in general and would have bought anyway (selection bias).
- The identical purchase rates despite very different view rates suggest that viewing is not the main driver of purchasing.
- The view-rate difference is likely linked to the social channel, but this was not tested.

### Assumptions
- A transaction belongs to an offer if it is by the same customer and falls between receipt and expiry.
- A transaction in the same hour as the first view counts as after the view.
- Purchase activity is measured over the full offer window for both groups.

### Limitations
- No control group: we cannot see what customers would have done without the offer.
- Transactions have no `offer_id`, so purchases cannot be tied to an offer directly.
- Purchase rate for informational offers is not comparable with completion rate for BOGO and discount offers. They measure different things.
- Only two informational offers exist, so conclusions about "informational offers" in general are limited.

### Supported recommendation
The data does not show that informational offers increase sales. A proper test with a holdout group would be needed to measure their effect.